# 第8回　仮説検定(1)：帰無仮説とt検定
## ―― 「差がある」とは、どういうことか

統計学Ⅰ（B）　／　北星学園大学

注目は、このコース屈指の誤解されやすい数 ――

> **p値は「効果の大きさ」でも「正しさの確率」でもない。**

### フック

> 保護区の中と外で、同じ種のサルの体重を測った。**保護区の中のほうが平均0.2kg重かった。**
> 
> これは「**保護区に効果があった**」のか、それとも「**たまたま**」なのか？

直感では決められない。0.2kgは大きい？小さい？――この問いに答えるのが**仮説検定**だ。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    # 原典（PanTHERIA）から組み立て直す。まずリポジトリ同梱の複製、だめなら発行元から。
    # 発行元は User-Agent を見て弾くことがあるため、明示して取得する。
    import io, urllib.request
    SRCS = [
        "https://raw.githubusercontent.com/aonoa68/toukei-1/main/docs/data/PanTHERIA_1-0_WR05_Aug2008.txt.gz",
        "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt",
    ]
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = None
    for _url in SRCS:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=60) as _r:
                _raw = _r.read()
            _comp = "gzip" if _url.endswith(".gz") else None
            src = pd.read_csv(io.BytesIO(_raw), sep="\t", compression=_comp)
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError("原典データを取得できませんでした")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. 帰無仮説 ―― 背理法で考える

仮説検定は**背理法**に似ている。

1. まず「**差はない（偶然だ）**」と仮定する。これを **帰無仮説** という。
2. その仮定のもとで、**観測されたような差が起きる確率**を計算する。これが **p値**。
3. p値がとても小さい（＝偶然ではめったに起きない）なら、「差はない」という仮定のほうが間違っていた、と考える。これを「**有意差がある**」という。

> 「差がある」＝「**偶然では説明しにくいほどの差だった**」の言い換え。

---
## 2. t検定をやってみる（明確な差の例）

霊長類の進化を語るとき、大きな区分が **旧世界ザル**（アフリカ・アジア）と**新世界ザル**（中南米）である。両者は3,000万年以上前に分かれた。

- **オナガザル科**（旧世界）… ニホンザル、ヒヒ、コロブスなど
- **オマキザル科**（新世界）… オマキザル、リスザルなど

この2つの科で**体重に差があるか**を調べる。2群の平均の差が偶然かを調べるのが**t検定**（`scipy.stats.ttest_ind`）。第3回のとおり体重は対数で扱う。

In [ ]:
旧世界 = np.log10(df[df["科"] == "オナガザル科"]["体重g"].dropna())
新世界 = np.log10(df[df["科"] == "オマキザル科"]["体重g"].dropna())

旧生 = df[df["科"] == "オナガザル科"]["体重g"].dropna()
新生 = df[df["科"] == "オマキザル科"]["体重g"].dropna()
print(f"オナガザル科（旧世界）: 中央値 {旧生.median():>7,.0f} g  (n={len(旧世界)})")
print(f"オマキザル科（新世界）: 中央値 {新生.median():>7,.0f} g  (n={len(新世界)})")
print(f"　　　　　　　　　　　  → 中央値の比 {旧生.median()/新生.median():.1f} 倍")

t, p = stats.ttest_ind(旧世界, 新世界, equal_var=False)
print(f"\nt値 = {t:.2f}")
print(f"p値 = {p:.2e}")
print("→ p値は極めて小さい（0.05よりはるかに下）。")
print("  『差はない（偶然）』ではこの差はめったに起きない → 有意差あり。")

p値はほぼ0。「旧世界ザルと新世界ザルで体重に差はない」と仮定すると、これほどの差（中央値で約15倍）は偶然ではまず起きない。だから**有意差あり**と判断する。

### 対照：差がない例も、同じデータの中にある

今度は **オナガザル科（旧世界）と クモザル科（新世界）** を比べる。クモザル科も新世界ザルだが、こちらは大型の種が多い。

In [ ]:
クモザル = np.log10(df[df["科"] == "クモザル科"]["体重g"].dropna())
クモ生 = df[df["科"] == "クモザル科"]["体重g"].dropna()

print(f"オナガザル科: 中央値 {旧生.median():>7,.0f} g  (n={len(旧世界)})")
print(f"クモザル科　: 中央値 {クモ生.median():>7,.0f} g  (n={len(クモザル)})")

t2, p2 = stats.ttest_ind(旧世界, クモザル, equal_var=False)
print(f"\nt値 = {t2:.2f}")
print(f"p値 = {p2:.3f}")
print("→ p値は0.98。まったく差があると言えない。")

**t = −0.02、p = 0.98。** ほとんど完全に一致している。

旧世界か新世界かで決まるのではない。**新世界ザルの中に、小型のオマキザル科と大型のクモザル科が両方いる**からである。

> **同じデータ・同じ手法で、比べる相手を変えただけで結論が正反対になる。**
> 「何と何を比べるか」は、検定にかける前の**あなたの判断**である。
> 統計は「その比較が妥当か」を教えてはくれない。

---
## 3. p値の意味を、正しく

ここで誤解を3つ、はっきり打ち消す。

- ❌ **p値は「帰無仮説が正しい確率」ではない**。p値は「帰無仮説が正しいと**仮定したとき**に、観測以上の差が出る確率」。
- ❌ **p値は「効果の大きさ」ではない**。p=0.001 でも差は小さいことがある。
- ❌ **「有意（p<0.05）＝重要」ではない**。有意は「偶然では説明しにくい」だけ。

これを、次のシミュレーションで体感する。

---
## 4. p値の脆さ ―― 同じ差でも n で変わる

冒頭のフックに戻る。保護区の外6.0kg・中6.2kg、**真の差はたった0.2kg**で固定。ここから n 頭ずつ捕獲して測る、を2000回くりかえし、**有意（p<0.05）になった割合**を見る。

効果（0.2kg）は同じなのに、n を増やすと…？

In [ ]:
rng = np.random.default_rng(2026)
mu1, mu2, sd = 6.0, 6.2, 1.1   # 真の差は0.2kgで固定（kg単位）

for n in [20, 100, 500]:
    有意 = 0
    for _ in range(2000):
        a = rng.normal(mu1, sd, n)
        b = rng.normal(mu2, sd, n)
        if stats.ttest_ind(a, b)[1] < 0.05:
            有意 += 1
    print(f"n={n:3d}頭/群: 有意(p<0.05)になった割合 = {有意/2000*100:3.0f}%")

print("\n真の差は常に0.2kg。なのに n を増やすほど『有意』になりやすい。")
print("→ 『有意』は効果の大きさではない。n を増やせば小さな差でも有意にできる。")

n=20 では1割ほどしか有意にならないのに、n=500 では大半が有意になる。**真の効果（0.2kg）は同じ**なのに、サンプルを増やすだけで「有意差あり」を作れてしまう。

だから「p<0.05だった！」だけでは、**効果が大きいことの証明にはならない**。効果の大きさは、別の指標で測る。

---
## 5. 効果量（Cohen's d）―― 「大きさ」を測る

p値とは別に、差の大きさ自体を標準化したのが **効果量 d**。「差が標準偏差いくつ分か」。目安：0.2小・0.5中・0.8大。

In [ ]:
def cohen_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.std(ddof=1)**2 + (nb-1)*b.std(ddof=1)**2) / (na+nb-2))
    return (a.mean() - b.mean()) / sp

d1 = cohen_d(旧世界, 新世界)
d2 = cohen_d(旧世界, クモザル)
d3 = (6.2 - 6.0) / 1.1

print(f"旧世界 vs 新世界（オマキザル科）の効果量 d = {d1:>6.2f}  → けた外れに大きい")
print(f"旧世界 vs クモザル科　　　　　　の効果量 d = {d2:>6.2f}  → 実質ゼロ")
print(f"保護区の内外（0.2kgの差）の効果量　　 d = {d3:>6.2f}  → 小さい効果")
print()
print("保護区の差は『n を増やせば有意にできる』が、効果量は小さい。有意 ≠ 効果が大きい。")

**d = 4.40。** 目安の「0.8で大」をはるかに超えている。旧世界ザルと新世界ザル（オマキザル科）の体重は、群内のばらつき4個分以上も離れている。

一方、保護区の例は d = 0.18。**n さえ増やせば有意にできるが、生物学的にはごくわずかな差**である。

> **p値は「偶然かどうか」、効果量は「どれくらいか」。**
> 論文でも報告書でも、**両方を書く**。片方だけでは判断できない。

---
## 6. 2種類の誤り

検定は完璧ではない。間違え方が2つある。

| | 本当は差がない | 本当は差がある |
|---|---|---|
| 「差あり」と判定 | **第一種の誤り（偽陽性）** | 正解 |
| 「差なし」と判定 | 正解 | **第二種の誤り（見逃し）** |

- **有意水準 α＝0.05**：本当は差がないのに「差あり」と誤る確率を5%まで許す、という基準。
- だから「有意」でも、**20回に1回は偶然の偽陽性**かもしれない。何度も検定すれば偽陽性は増える（→第9回）。

> §4で見た「n=20では1割しか有意にならない」は、**第二種の誤り（見逃し）**が9割起きているということでもある。**差はあるのに、検出できていない。**

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 帰無仮説 | まず「差はない（偶然）」と仮定する |
| p値 | 帰無仮説のもとで観測以上の差が出る確率 |
| 有意差 | p<0.05 ＝「偶然では説明しにくい」 |
| ❌ よくある誤り | p値=帰無仮説が正しい確率／有意=効果大／n増やせば必ず有意 |
| 効果量 d | 差の大きさ。有意とは別に必ず見る（旧vs新世界で d=4.40） |
| α・2種の誤り | 偽陽性5%を許す基準。検定の繰り返しに注意 |
| **比較の設計** | **何と何を比べるかは検定の外側。統計は妥当性を教えない** |

> **「差がある」＝「偶然では説明しにくい」。それ以上でも以下でもない。**
> p値の小ささは効果の大きさを意味しない。**必ず効果量も見る。**

**課題（Moodle）**：t検定の出力を解釈し、「p<0.05だから効果が大きい」という主張の誤りを指摘する。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。